In [11]:
import numpy as np
from centerline_pipeline import extract_centerline
import matplotlib.pyplot as plt 
from scipy.interpolate import interp1d
from matplotlib.animation import FuncAnimation, PillowWriter

G = np.arange(1, 5+1)
Tf = 5
times = np.arange(1, Tf+1)
N = 100

for g in G: 
    X = np.zeros((Tf, N, 3))
    for i, t in enumerate(times):
        X[i] = extract_centerline(g, t, N)
        
        fx = interp1d(times, X[:,:,0], axis=0, kind='cubic')
        fy = interp1d(times, X[:,:,1], axis=0, kind='cubic')
        fz = interp1d(times, X[:,:,2], axis=0, kind='cubic')

        times_fine = np.linspace(1, 5, 41)

        Xfine = np.stack([
            fx(times_fine),
            fy(times_fine),
            fz(times_fine)
        ], axis=2)

        Xfine.shape

        np.savez(
            f"../../data/synthetic/synthetic_curves/gen_{g:02d}/dodder_centerline_g{g:02d}.npz",
            X = X,
            Xfine = Xfine,
            times = times,
            times_fine = times_fine,
            N = N,
            g = g
        )
        fig = plt.figure(figsize=(8,8))
        ax = fig.add_subplot(111, projection='3d')

        # Fix axis limits so the view doesn't jump
        ax.set_xlim(Xfine[:,:,0].min(), Xfine[:,:,0].max())
        ax.set_ylim(Xfine[:,:,1].min(), Xfine[:,:,1].max())
        ax.set_zlim(Xfine[:,:,2].min(), Xfine[:,:,2].max())

        line, = ax.plot([], [], [], 'r-', lw=2)

        anim = FuncAnimation(
            fig,
            update,
            frames=len(times_fine),
            interval=100,
            blit=False
        )

        anim.save(
            f"../../data/synthetic/synthetic_curves/gen_{g:02d}/dodder_growth_g{g:02d}.gif",
            writer=PillowWriter(fps=10)
        )

        plt.close()

def update(frame):

    curve = Xfine[frame]

    line.set_data(
        curve[:,0],
        curve[:,1]
    )

    line.set_3d_properties(
        curve[:,2]
    )

    ax.set_title(f"t = {times_fine[frame]:.2f}")

    return line,